# Train CHURN_MODEL (DEV)

Deliberately minimal example notebook for the sf-mlops DEV → QA → PROD pipeline.

- Trains a simple `LogisticRegression` on `MLOPS_DEV.MLOPS.TRAIN_DATA` (synthetic).
- Registers it to the Model Registry in `MLOPS_DEV.MLOPS` with a timestamped version name.
- Moves the `DEV` alias to this new version — the QA workflow simply reads whichever version currently holds that alias.

Note: if two people train around the same time, whoever moves the `DEV` alias last "wins" — there's no locking/collision handling here by design; handle that at the process level if it becomes an issue.

In [ ]:
from datetime import datetime

from snowflake.ml.registry import Registry
from snowflake.snowpark.context import get_active_session
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

MODEL_NAME = "CHURN_MODEL"
FEATURE_COLUMNS = ["FEATURE_1", "FEATURE_2", "FEATURE_3", "FEATURE_4", "FEATURE_5", "FEATURE_6"]
LABEL_COLUMN = "LABEL"

session = get_active_session()

In [ ]:
# Load the synthetic training data and split off a local validation slice
# (this is just a sanity-check metric for the developer -- the real gate is
# the QA evaluation against the holdout set, which this role can't even see).
train_pdf = session.table("MLOPS_DEV.MLOPS.TRAIN_DATA").to_pandas()

X = train_pdf[:100][FEATURE_COLUMNS]
y = train_pdf[:100][LABEL_COLUMN]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} rows, validating on {len(X_valid)} rows")

In [ ]:
clf = LogisticRegression(max_iter=100)
clf.fit(X_train, y_train)

train_accuracy = accuracy_score(y_valid, clf.predict(X_valid))
print(f"Local validation accuracy: {train_accuracy:.4f}")

In [ ]:
%%sql -r dataframe_1
USE SCHEMA MLOPS_DEV.MLOPS;

In [ ]:
version_name = f"DEV_{datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

reg = Registry(session=session, database_name="MLOPS_DEV", schema_name="MLOPS")
mv = reg.log_model(
    clf,
    model_name=MODEL_NAME,
    version_name=version_name,
    conda_dependencies=["scikit-learn"],
    comment="Trained in notebooks/train_model.ipynb",
    metrics={"train_accuracy": train_accuracy},
    sample_input_data=X_train,
    target_platforms=["WAREHOUSE"]
)
print(f"Logged {MODEL_NAME} version {version_name}")

In [ ]:
# Move the DEV alias to this version -- this is the ONLY thing the QA
# workflow reads to decide what to evaluate next.
#
# We use the Model Registry Python API (model.show_versions(), a pandas
# DataFrame) to find who currently holds the alias, rather than parsing
# `SHOW VERSIONS IN MODEL` SQL output -- simpler, and avoids a SQL type
# mismatch (RESULT_SCAN surfaces "aliases" as a JSON-encoded VARCHAR, not a
# native ARRAY, so ARRAY_CONTAINS on it directly fails to compile).
import json

fq_model = f"MLOPS_DEV.MLOPS.{MODEL_NAME}"
versions_df = reg.get_model(MODEL_NAME).show_versions()

current_dev_version = None
for _, row in versions_df.iterrows():
    aliases = row["aliases"]
    if isinstance(aliases, str):
        try:
            aliases = json.loads(aliases)
        except (TypeError, ValueError):
            aliases = []
    if aliases and "DEV" in aliases:
        current_dev_version = row["name"]
        break

if current_dev_version is not None:
    session.sql(f"ALTER MODEL {fq_model} VERSION {current_dev_version} UNSET ALIAS").collect()
session.sql(f"ALTER MODEL {fq_model} VERSION {version_name} SET ALIAS = DEV").collect()

print(f"DEV alias -> {version_name}")

# EOL